In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from utils import *
from llm import llm_call

In [30]:
dev_nn = load_json('bert_neighbors_500/dev_nearest_examples.json')
dev_data = load_json('data/source/mrbench_v3_devset.json')

test_nn = load_json('bert_neighbors_500/test_nearest_examples.json')
test_data = load_json('data/source/mrbench_v3_testset.json')

In [77]:
def get_example_from_data(data, idx):
    # search data list for the example with the given index
    for example in data:
        if example['conversation_id'] == idx:
            return example
        
def get_nns_dev(data, idx, tutor):
    # search data list for the example with the given index
    iid = idx + 'SEP' + tutor
    for example in data:
        if example['dev_id'] == iid:
            return example
        

def get_nns_test(data, idx, tutor):
    # search data list for the example with the given index
    iid = idx + 'SEP' + tutor
    for example in data:
        if example['test_id'] == iid:
            return example


In [98]:
def get_few_shot_examples_dev(example, tutor):
    eid = example['conversation_id']
    selected_startified_nn = []
    labels_list=['Yes', 'To some extent', 'No']
    label_groups = {label: 0 for label in labels_list}
    for tutor_id, tutor_info in example['tutor_responses'].items():
        if tutor_id != tutor:
            continue
        nns = get_nns_dev(dev_nn, eid, tutor_id)
        print(f"there are {len(nns['nearest_examples'])} nearest examples")
        for nn in nns['nearest_examples'][1:]:
            nn_id, nn_tutor = nn['id'].split('SEP')
            nn_example = get_example_from_data(dev_data, nn_id)
            nn_label = nn_example['tutor_responses'][nn_tutor]['annotation']['Mistake_Identification']
            if label_groups[nn_label] < 3:
                path = f"./cot_t1/{nn['id'].replace('SEP', '_')}.json"
                cot = load_json(path)
                selected_startified_nn.append(cot['one_shot'])
                label_groups[nn_label] += 1
    return selected_startified_nn


def get_few_shot_examples_test(example, tutor):
    eid = example['conversation_id']
    selected_startified_nn = []
    labels_list=['Yes', 'To some extent', 'No']
    label_groups = {label: 0 for label in labels_list}
    for tutor_id, tutor_info in example['tutor_responses'].items():
        if tutor_id != tutor:
            continue
        nns = get_nns_test(test_nn, eid, tutor_id)
        print(f"there are {len(nns['nearest_examples'])} nearest examples")
        for nn in nns['nearest_examples'][1:]:
            nn_id = nn['id']
            nn_tutor = nn['label']
            nn_example = get_example_from_data(dev_data, nn_id)
            nn_label = nn_example['tutor_responses'][nn_tutor]['annotation']['Mistake_Identification']
            if label_groups[nn_label] < 3:
                path = f"./cot_t1/{nn_id}_{nn_tutor}.json"
                cot = load_json(path)
                selected_startified_nn.append(cot['one_shot'])
                label_groups[nn_label] += 1
    return selected_startified_nn

In [99]:
selected_startified_nn = get_few_shot_examples_dev(dev_data[0], 'Sonnet')
selected_startified_nn

there are 500 nearest examples


['\n### Example:\n### Dialogue:\n- Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: If Marcy works for the same company for 40 years, she gets an annual pension of $50,000/year. Starting after 20 years, she becomes entitled to 5% of the value of the pension per year. If she quits after 30 years, what will her annual pension be?\n- Student: Marcy is entitled to 5% x 20 = 100% of the value of the pension.\nSo, her annual pension after 20 years is $50,000 x 100% = $50,000.\nFor the next 10 years, Marcy is entitled to an additional 5% x 10 = 50% of the value of the pension.\nSo, her annual pension after 30 years is $50,000 + ($50,000 x 50%) = $50,000 + $25,000 = $75,000.\n\xa075,000\n- Tutor: Winnie, According to the problem, how many years does Marcy have to work before she earns ANY pension?\n- Student: Marcy has to work for at least 20 years before she earns any pension.\n- Tutor: How many years past 20 does she work if she works there

In [100]:
selected_startified_nn = get_few_shot_examples_test(test_data[0], 'Tutor_1')
selected_startified_nn

there are 500 nearest examples


['\n### Example:\n### Dialogue:\n- Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: A shady restaurant is charging customers gratuities after taxes without them being aware. If my total bill was $140, the sales tax in my city is 10%, I ordered a NY Striploin for $80, and I ordered a glass of wine for $10, how much did they charge me for gratuities?\n- Student: The total cost of food and drinks before taxes and gratuities is 80+10 = $90.\nThe tax on the food and drinks is 10/100 x 90 = $9.\nThe total bill including tax is 90+9 = $99.\nThe gratuity is charged on the total bill, so the amount of gratuity is 10/100 x 140 = $14.\nTherefore, the shady restaurant charged $14 for gratuities.\n 14\n- Tutor: could you please walk me through your solution?\n- Student: Sure. The total cost of food and drinks before taxes and gratuities is 80+10 = $90. Then the tax on the food and drinks is 10/100 x 90 = $9. The total bill including tax is 90+9 = 

In [104]:
print(test_data[0]['conversation_history'])

print('>>',test_data[0]['tutor_responses']['Tutor_1']['response'])

Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Tyson decided to make muffaletta sandwiches for the big game.  Each sandwich required 1 pound each of meat and cheese and would serve 4 people.  There would be 20 people in total watching the game.  The meat cost $7.00 per pound and the cheese cost $3.00 per pound.  How much money would he spend on the meat and cheese to make enough sandwiches to serve 20 people? 
 Student: To serve 20 people, Tyson needs to make 20/4 = 5 sandwiches.
Each sandwich requires 1+1 = 2 pounds of meat and cheese.
For 5 sandwiches, he needs a total of 2 x 5 = 10 pounds of meat and cheese.
The cost of 10 pounds of meat is 10 x $7.00 = $70.
The cost of 10 pounds of cheese is 10 x $3.00 = $30.
The total cost of meat and cheese is $70 + $30 = $100.
 100 
 Tutor: do you want to talk me through your solution 
 Student: Yes I think my answer is correct. I used 10 pounds of meat and 10 pounds of cheese so I multiplied

In [106]:
for kk in selected_startified_nn:
    print(kk)
    print('---'*10)


### Example:
### Dialogue:
- Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: A shady restaurant is charging customers gratuities after taxes without them being aware. If my total bill was $140, the sales tax in my city is 10%, I ordered a NY Striploin for $80, and I ordered a glass of wine for $10, how much did they charge me for gratuities?
- Student: The total cost of food and drinks before taxes and gratuities is 80+10 = $90.
The tax on the food and drinks is 10/100 x 90 = $9.
The total bill including tax is 90+9 = $99.
The gratuity is charged on the total bill, so the amount of gratuity is 10/100 x 140 = $14.
Therefore, the shady restaurant charged $14 for gratuities.
 14
- Tutor: could you please walk me through your solution?
- Student: Sure. The total cost of food and drinks before taxes and gratuities is 80+10 = $90. Then the tax on the food and drinks is 10/100 x 90 = $9. The total bill including tax is 90+9 = $99. Finally 

In [66]:
for example in dev_data:
    eid = example['conversation_id']
    analysis = load_json(f'correct_solutions/{eid}.json')

    for tutor_id, tutor_info in example['tutor_responses'].items():
        nns = get_nns_dev(dev_nn, eid, tutor_id)
        selected_startified_nn = []
        label_groups = {label: 0 for label in labels_list}
        for nn in nns['nearest_examples'][1:]:
            nn_id, nn_tutor = nn['id'].split('SEP')
            nn_example = get_example_from_data(dev_data, nn_id)
            nn_label = nn_example['tutor_responses'][nn_tutor]['annotation']['Mistake_Identification']
            if label_groups[nn_label] < 3:
                path = f"./cot_t1/{nn['id'].replace('SEP', '_')}.json"
                cot = load_json(path)
                selected_startified_nn.append(cot['one_shot'])
                label_groups[nn_label] += 1
        break
    break


In [67]:
len(nns['nearest_examples'])

500

In [68]:
label_groups, tutor_id

({'Yes': 3, 'To some extent': 3, 'No': 3}, 'Sonnet')

In [70]:
selected_startified_nn

['\n### Example:\n### Dialogue:\n- Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: If Marcy works for the same company for 40 years, she gets an annual pension of $50,000/year. Starting after 20 years, she becomes entitled to 5% of the value of the pension per year. If she quits after 30 years, what will her annual pension be?\n- Student: Marcy is entitled to 5% x 20 = 100% of the value of the pension.\nSo, her annual pension after 20 years is $50,000 x 100% = $50,000.\nFor the next 10 years, Marcy is entitled to an additional 5% x 10 = 50% of the value of the pension.\nSo, her annual pension after 30 years is $50,000 + ($50,000 x 50%) = $50,000 + $25,000 = $75,000.\n\xa075,000\n- Tutor: Winnie, According to the problem, how many years does Marcy have to work before she earns ANY pension?\n- Student: Marcy has to work for at least 20 years before she earns any pension.\n- Tutor: How many years past 20 does she work if she works there

In [107]:
get_example_from_data(dev_data, '4220-7d6b7aaa-7c4c-4dbb-a019-9ef1c213a087')

{'conversation_id': '4220-7d6b7aaa-7c4c-4dbb-a019-9ef1c213a087',
 'conversation_history': 'Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Emily is 20 years old and her older sister, Rachel, is 24 years old.\xa0\xa0How old is Rachel when Emily is half her age?\xa0\n\xa0Student: Let\'s call the age Rachel is when Emily is half her age "x".\xa0\n\nWhen Emily is half Rachel\'s age, Emily will be 20 + x years old.\xa0\n\nWe know that when Emily is half Rachel\'s age:\xa0\n\n20 + x = 1/2 (24 + x)\xa0\n\nMultiplying both sides by 2:\xa0\n\n40 + 2x = 24 + x\xa0\n\nSubtracting x and 24 from both sides:\xa0\n\n16 = x\xa0\n\nSo Rachel is 24 + x = 40 years old when Emily is half her age.\xa0\n\n\xa040\xa0\n\xa0Tutor: can you tell me how you got your answer?\xa0\n\xa0Student: Sure. I used the equation 20 + x = 1/2 (24 + x) and then multiplied both sides by 2. Then I subtracted x and 24 from both sides and got 16 = x. So Rachel is 24 + x = 40 yea